In [226]:

from langsmith import Client
from pathlib import Path

from dotenv import load_dotenv

import langsmith as ls
import os

In [227]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition


from typing import TypedDict, Literal, Annotated
from typing_extensions import NotRequired

from pydantic import BaseModel, Field

In [228]:
# Load env vars before any LangChain import (tracing caches the client on first import).
project_root = Path.cwd()
if project_root.name == "src":
    project_root = project_root.parent
elif not (project_root / ".env").exists() and (project_root.parent / ".env").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env", override=True)

# Reset LangSmith's cached tracing client so it picks up the fresh API key.
ls.configure(client=Client())

In [229]:
openai_key = os.getenv("OPENAI_KEY")

In [230]:
Client().list_projects(limit=1)

<generator object Client.list_projects at 0x000002D99F2B9FC0>

In [231]:
llm = ChatOpenAI(
    model ="gpt-4.1-mini",
    temperature =0,
    api_key = openai_key
)

In [232]:
@tool
def add(a:float, b:float)->float:
    """ This tool adds two numbers """
    return a+b

@tool
def multiply(a:float, b:float)->float:
    """ This tool multiples two numbers"""
    return a*b

@tool
def divide(a:float, b:float)->float:
    """ This tool divides two number"""
    if(a>b):
        return b/a
    else:
        return a/b

In [233]:
llm_with_tools = llm.bind_tools([add, multiply, divide])

In [234]:
# state

class State(BaseModel):
    messages:Annotated[list[BaseMessage], add_messages]
    user_query:str
    total_calls : int = Field(description="tells how many times llm has been called", default=0)

In [235]:
def reason_and_respond(state:State):
    messages =[]
    if not state.messages:
        messages = [
            HumanMessage(content=state.user_query)
        ]
    else:
        messages = state.messages

    response = llm_with_tools.invoke(messages)

    return {
        "total_calls":state.total_calls +1,
        "messages":[response]
    }

In [236]:
tools = [add, multiply, divide]
tool_node = ToolNode(tools)

In [237]:
def should_continue(state: State)->["tools", "end"]:

    last_message =  state.messages[-1]

    if last_message.tool_calls:
        return "tools"

    return "end"

In [238]:
# graph 


In [239]:
graph = StateGraph(State)

In [240]:
graph.add_node("reason_and_respond", reason_and_respond)
graph.add_node("execute", tool_node)

In [241]:
graph.add_edge(START, "reason_and_respond")
graph.add_conditional_edges(
    "reason_and_respond",
    should_continue,
    {
        "tools":"execute",
        "end":END
    }
)
graph.add_edge("execute", "reason_and_respond")

In [242]:
workflow = graph.compile()

In [244]:
initial_state ={
    "user_query": "what be will 18 added to 76. Multiply the result by 4. Divide it by 3"
}
result = workflow.invoke(initial_state)
print(result)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 114, 'total_tokens': 181, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_d525840b87', 'id': 'chatcmpl-EF9arC7Jt4Sn3xRLC7U1eihhQCsOF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a02238-2f5c-7791-b0df-d0888b3dca40-0', tool_calls=[{'name': 'add', 'args': {'a': 18, 'b': 76}, 'id': 'call_peNoV7IoPKV0vGcnJy1JIkSs', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 94, 'b': 4}, 'id': 'call_tN8WgaqpsMM9ZQAfE1psNVc1', 'type': 'tool_call'}, {'name': 'divi